In [1]:
%cd ..

/home/blanka/Multi-Domain-Pruning


/data/blanka/virtualenvs/yolov10/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import pandas as pd
import numpy as np
import sys

from src.model.model_handler import ModelHandler
from src.sample_handler import SampleHandler
from utils.config_parser import ConfigParser
from pruning.channel_selection.channel_selector import ChannelSelector
from pruning.model_pruner.stepwise_pruner import StepWisePruner

/data/blanka/virtualenvs/yolov10/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
alpha_sequence = np.linspace(0, 2.2, 93).round(1)

In [ ]:
# Read and save config file
conf = ConfigParser.read("config/pruning/pruning_sampling.ini")
conf.model.batch_size = 4

model_handler = ModelHandler(conf.model)
channel_selector = ChannelSelector(conf.channel_selection)

# Determine prunable layers
model_handler.determine_prunable_layers()
assert len(alpha_sequence) == model_handler.n_prunable_layers, (
f"Alpha_sequence length ({len(alpha_sequence)}) is not equal to the number of prunable layers ({model_handler.n_prunable_layers})!"
)

# Evaluate initial model
print("\n\n########## INITIAL MODEL ##########\n\n")
model_handler.evaluate()

# Collect initial model layer info
init_model_channels = [layer[1].out_channels for layer in model_handler.prunable_layers]

# Select indices and prune model layer-wise
all_indices = [None] * model_handler.n_prunable_layers
for i, layer in enumerate(model_handler.prunable_layers):
    idxs = channel_selector.select_indices(layer[1], alpha_sequence[i])
    all_indices[i] = idxs
    model_handler.prune(all_indices, i)
    model_handler.determine_prunable_layers()

# Collect pruned model layer info
pruned_model_channels = [layer[1].out_channels for layer in model_handler.prunable_layers]

# Print statistics
for i, layer in enumerate(model_handler.prunable_layers):
    n_removed_channels = init_model_channels[i] - pruned_model_channels[i]
    print(f"Layer {i:<3} {layer[0]:<20} a = {alpha_sequence[i]:<5} {n_removed_channels:<3}/{init_model_channels[i]:<5} channels removed.")

# Eval pruned model
print("\n\n########## PRUNED MODEL ##########\n\n")
model_handler.evaluate()
